# 03 — DistilBERT Fine-Tuning for SMS Spam Detection

## 1. Project Objective

The objective of this notebook is to fine-tune a pretrained DistilBERT transformer model to classify SMS messages as either legitimate (ham) or spam.

This experiment builds on the initial exploratory data analysis and the TF-IDF + Logistic Regression baseline developed earlier in the project.Untitled

## 2. Dataset

- **Source:** UCI SMS Spam Collection
- **Task:** Binary text classification
- **Input:** SMS message text
- **Target:** Message label (`ham` or `spam`)
- **Class mapping:** `ham = 0`, `spam = 1`
- **Dataset size after cleaning:** 5,169 messages

## 3. Experimental Workflow

1. Import required libraries and verify the environment.
2. Load the original dataset and reproduce the established cleaning procedure.
3. Split the dataset into training, validation, and test sets using stratified sampling.
4. Encode the target labels as numeric class IDs.
5. Load the pretrained DistilBERT tokenizer.
6. Tokenize the SMS messages and prepare input tensors.
7. Load the pretrained DistilBERT model with a sequence-classification head.
8. Fine-tune the model using the training set.
9. Monitor validation performance and select the best model checkpoint.
10. Evaluate the selected model on the held-out test set.
11. Compare the results with the TF-IDF + Logistic Regression baseline.
12. Save the trained model and tokenizer for later application integration.

## 4. Evaluation Metrics

The model will be evaluated using:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix

Particular attention will be given to spam recall and spam precision because both missed spam messages and legitimate messages incorrectly flagged as spam matter in a practical spam detector.

## 5. Reproducibility and Data Integrity

- Use a fixed random seed for reproducible data splitting and training where applicable.
- Use stratified sampling to preserve class proportions.
- Keep training, validation, and test data separate.
- Use the validation set for model-development decisions.
- Reserve the test set for final evaluation.
- Keep raw and processed datasets out of version control when excluded by `.gitignore`.
- Record model configuration, training settings, and evaluation results.

## 6. Expected Outcome

A fine-tuned DistilBERT spam classifier, a documented evaluation against the classical machine-learning baseline, and saved model artifacts that can later be integrated into a FastAPI application.


In [1]:
# Python standard library
import random
import json
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Deep learning
import torch
from torch.utils.data import Dataset, DataLoader

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Hugging Face Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)


In [2]:
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

print("\nAll packages imported successfully!")

PyTorch: 2.15.0+cpu
NumPy: 2.5.2
Pandas: 3.0.5
CUDA build: None
CUDA available: False

All packages imported successfully!


In [3]:
# Locate  raw dataset
data_path = Path("../data/raw/SMSSpamCollection")

# whether the file exists
print("Dataset exists:", data_path.exists())
print("Dataset path:", data_path.resolve())

Dataset exists: True
Dataset path: /home/student/email-spam-detector/data/raw/SMSSpamCollection


In [41]:
df = pd.read_csv(
    data_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("Dataset shape:", df.shape)
print("\nFirst five rows:")

print("Original dataset:", len(df))
print("Duplicate rows:", df.duplicated().sum())
display(df.head())

Dataset shape: (5572, 2)

First five rows:
Original dataset: 5572
Duplicate rows: 403


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [43]:
duplicates = df[df.duplicated(keep=False)]

duplicate_label_counts = (
    duplicates.groupby("message")["label"]
    .nunique()
)

print(duplicate_label_counts.value_counts())

label
1    281
Name: count, dtype: int64


In [44]:
# Remove exact duplicate rows
df_clean = df.drop_duplicates().reset_index(drop=True)

# Verify cleaned dataset
print("Original dataset:", len(df))
print("Cleaned dataset:", len(df_clean))
print("Remaining duplicates:", df_clean.duplicated().sum())

Original dataset: 5572
Cleaned dataset: 5169
Remaining duplicates: 0


In [45]:
X = df_clean["message"]
y = df_clean["label"]

print("Messages:", X.shape)
print("Labels:", y.shape)

Messages: (5169,)
Labels: (5169,)


In [48]:
# Convert text labels into numerical labels
label_mapping = {
    "ham": 0,
    "spam": 1
}

#creates a new column containing those numerical labels
df_clean["label_id"] = df_clean["label"].map(label_mapping)

# Inspect the conversion
print(df_clean[["label", "label_id"]].head())

# Check for labels that were not mapped
print("\nMissing numerical labels:", df_clean["label_id"].isnull().sum())

  label  label_id
0   ham         0
1   ham         0
2  spam         1
3   ham         0
4   ham         0

Missing numerical labels: 0


In [54]:
X = df_clean["message"]
y = df_clean["label_id"]

print("Messages:", X.shape)
print("Labels:", y.shape)
print("Label values:", y.unique())

Messages: (5169,)
Labels: (5169,)
Label values: [0 1]


In [57]:
# First split: 70% training, 30% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: divide temporary data equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))
print("Total samples:", len(X_train) + len(X_val) + len(X_test))

Training samples: 3618
Validation samples: 775
Testing samples: 776
Total samples: 5169


In [58]:
# Check class proportions in each dataset split
print("Training distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nValidation distribution:")
print(y_val.value_counts(normalize=True).sort_index())

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True).sort_index())

Training distribution:
label_id
0    0.873687
1    0.126313
Name: proportion, dtype: float64

Validation distribution:
label_id
0    0.873548
1    0.126452
Name: proportion, dtype: float64

Testing distribution:
label_id
0    0.873711
1    0.126289
Name: proportion, dtype: float64


In [61]:
# Load the pretrained DistilBERT tokenizer
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

Tokenizer loaded successfully!
Vocabulary size: 30522


In [62]:
# Tokenize training messages
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    max_length=128
)

# Tokenize validation messages
val_encodings = tokenizer(
    X_val.tolist(),
    truncation=True,
    max_length=128
)

# Tokenize test messages
test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    max_length=128
)

In [63]:
print("Training messages:", len(train_encodings["input_ids"]))
print("Validation messages:", len(val_encodings["input_ids"]))
print("Test messages:", len(test_encodings["input_ids"]))

print("\nFirst training message token IDs:")
print(train_encodings["input_ids"][0])

print("\nFirst training message attention mask:")
print(train_encodings["attention_mask"][0])

Training messages: 3618
Validation messages: 775
Test messages: 776

First training message token IDs:
[101, 1045, 2228, 2057, 1005, 2128, 2183, 2000, 9303, 1005, 1055, 2085, 1010, 2272, 102]

First training message attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [64]:
class SMSDataset(Dataset):

    def __init__(self, encodings, labels):
        # Store the tokenized encodings in this object
        self.encodings = encodings

        # Convert labels to a Python list and store them
        self.labels = labels.tolist()

    def __len__(self):
        # Return the number of labels stored in this object
        return len(self.labels)

    def __getitem__(self, idx):
        # Retrieve the input IDs for the message at index idx
        input_ids = self.encodings["input_ids"][idx]

        # Retrieve the attention mask for the message at index idx
        attention_mask = self.encodings["attention_mask"][idx]

        # Retrieve the label for the message at index idx
        label = self.labels[idx]

        # Convert input IDs into a PyTorch tensor
        input_ids = torch.tensor(input_ids, dtype=torch.long)

        # Convert attention mask into a PyTorch tensor
        attention_mask = torch.tensor(attention_mask, dtype=torch.long)

        # Convert the label into a PyTorch tensor
        label = torch.tensor(label, dtype=torch.long)

        # Return a dictionary containing the tensors
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": label
        }

In [65]:
train_dataset = SMSDataset(train_encodings, y_train)

val_dataset = SMSDataset(val_encodings, y_val)

test_dataset = SMSDataset(test_encodings, y_test)

print(
    "Training length:", len(train_dataset),
    "\nValidation length:", len(val_dataset),
    "\nTesting length:", len(test_dataset)
)

Training length: 3618 
Validation length: 775 
Testing length: 776


In [66]:
sample = train_dataset[0]

print(sample.items())
print(sample.values())
print(sample.keys())

dict_items([('input_ids', tensor([ 101, 1045, 2228, 2057, 1005, 2128, 2183, 2000, 9303, 1005, 1055, 2085,
        1010, 2272,  102])), ('attention_mask', tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])), ('labels', tensor(0))])
dict_values([tensor([ 101, 1045, 2228, 2057, 1005, 2128, 2183, 2000, 9303, 1005, 1055, 2085,
        1010, 2272,  102]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]), tensor(0)])
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [67]:
print("Input IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)
print("Label shape:", sample["labels"].shape)

Input IDs shape: torch.Size([15])
Attention mask shape: torch.Size([15])
Label shape: torch.Size([])


In [68]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [69]:
data_collator 

DataCollatorWithPadding(tokenizer=BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

In [72]:
BATCH_SIZE = 16

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

In [73]:
batch = next(iter(train_dataloader))

print("\nIndividual values")
print(batch["input_ids"][:2])
print(batch["attention_mask"][:2])
print(batch["labels"][:2])
print("\nShapes")
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)


Individual values
tensor([[  101,  2123,  1005,  1056,  2228,  2055,  1000,  2054,  1057,  2031,
          2288,  1000,  2228,  2055,  1000,  2129,  2000,  2224,  2009,  2008,
          2017,  2031,  2288,  1000,  2204,  9152,  2620,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0],
        [  101, 16780,  1012,  5564,  2016,  2831,  2360,  2191,  1045, 17021,
          2017,  1012,   102,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1

In [74]:
print("Real tokens in message 1:", batch["attention_mask"][0].sum().item())
print("Real tokens in message 2:", batch["attention_mask"][1].sum().item())

Real tokens in message 1: 28
Real tokens in message 2: 13


In [75]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [76]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [77]:
print("Total parameters:", sum(p.numel() for p in model.parameters()))

print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

Total parameters: 66955010
Trainable parameters: 66955010


In [78]:
model.train()

outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"]
)

print(outputs.logits)
print(outputs.logits.shape)

tensor([[-0.1132, -0.0972],
        [-0.0996, -0.1578],
        [-0.1418, -0.1108],
        [-0.0051, -0.1907],
        [-0.0322, -0.0678],
        [-0.0357, -0.1124],
        [-0.0727, -0.0232],
        [-0.1343, -0.0914],
        [-0.0891, -0.1657],
        [-0.0155, -0.0288],
        [-0.1685, -0.0999],
        [-0.2035, -0.1502],
        [-0.0443, -0.1023],
        [-0.1425, -0.0327],
        [-0.0914, -0.0226],
        [-0.0400, -0.0746]], grad_fn=<AddmmBackward0>)
torch.Size([16, 2])


In [79]:
print(batch["labels"])

tensor([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0])


In [80]:
loss_function = torch.nn.CrossEntropyLoss()

In [81]:
loss = loss_function(outputs.logits, batch["labels"])

print("Loss:", loss.item())

Loss: 0.6833286285400391


In [82]:
### The computational sequence is:
# optimizer.zero_grad()  # Clear previous gradients

# outputs = model(...)   # Forward propagation

# loss = loss_function(...)  # Calculate loss

# loss.backward()        # Calculate gradients

# optimizer.step()       # Update model parameters

In [83]:
print(model.classifier.weight.grad) #Checking the gradient before backpropagation

None


In [84]:
loss.backward()


In [85]:
print(model.classifier.weight.grad.shape)

torch.Size([2, 768])


In [86]:
print(model.classifier.weight.grad)  # Inspecting the actual gradient values

tensor([[-0.0210,  0.0000, -0.0002,  ...,  0.0000,  0.0000, -0.0271],
        [ 0.0210,  0.0000,  0.0002,  ...,  0.0000,  0.0000,  0.0271]])


In [87]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=1e-2
)

In [88]:
optimizer.zero_grad() #clear gradient

In [89]:
print(model.classifier.weight.grad)

None


In [90]:
EPOCHS = 3   #3 means the model will process all 3,900 training messages three times

In [91]:
def train_one_epoch(model, dataloader, optimizer, loss_function):

    model.train()
    model.zero_grad(set_to_none=True)
    total_loss = 0

    for batch_number, batch in enumerate(dataloader, start=1):

        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        loss = loss_function(outputs.logits, batch["labels"])

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch["labels"].size(0)
        
        if batch_number % 20 == 0: # checks the remainder when the batch number is divided by 20
            print(
                f"Processed {batch_number}/{len(dataloader)} batches | "
                f"Batch Loss: {loss.item():.4f}"
            )

    average_loss = total_loss / len(dataloader.dataset)   #divides the accumulated loss by the number of examples

    return average_loss

In [43]:
training_loss = train_one_epoch(
    model,
    train_dataloader,
    optimizer,
    loss_function
)

print(f"\nAverage Training Loss: {training_loss:.4f}")

Processed 20/244 batches | Batch Loss: 0.2690
Processed 40/244 batches | Batch Loss: 0.0581
Processed 60/244 batches | Batch Loss: 0.0448
Processed 80/244 batches | Batch Loss: 0.0495
Processed 100/244 batches | Batch Loss: 0.0096
Processed 120/244 batches | Batch Loss: 0.0381
Processed 140/244 batches | Batch Loss: 0.0132
Processed 160/244 batches | Batch Loss: 0.0516
Processed 180/244 batches | Batch Loss: 0.0090
Processed 200/244 batches | Batch Loss: 0.0040
Processed 220/244 batches | Batch Loss: 0.0032
Processed 240/244 batches | Batch Loss: 0.0045

Average Training Loss: 0.0879


In [92]:
def evaluate_model(model, dataloader, loss_function):
    model.eval()
    
    total_loss = 0
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )

            loss = loss_function(outputs.logits, batch["labels"])

            total_loss += loss.item() * batch["labels"].size(0)

            batch_predictions = torch.argmax(outputs.logits, dim=1)
            
            predictions.extend(batch_predictions.cpu().tolist())
            true_labels.extend(batch["labels"].cpu().tolist())
            
    average_loss = total_loss / len(dataloader.dataset)
    
    return average_loss, predictions, true_labels

In [46]:
validation_loss, predictions, true_labels = evaluate_model(
    model,
    val_dataloader,
    loss_function               #python unpacking
)

print(f"Validation Loss: {validation_loss:.4f}")


Validation Loss: 0.0268


In [47]:
accuracy = accuracy_score(true_labels, predictions)

precision = precision_score(true_labels, predictions, zero_division=0)

recall = recall_score(true_labels, predictions, zero_division=0)

f1 = f1_score(true_labels, predictions, zero_division=0)

In [48]:
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

Accuracy:  0.9916
Precision: 0.9565
Recall:    0.9821
F1-score:  0.9692


In [49]:
training_loss = train_one_epoch(
    model,
    train_dataloader,
    optimizer,
    loss_function
)

print(f"\nEpoch 2 Training Loss: {training_loss:.4f}")

Processed 20/244 batches | Batch Loss: 0.0053
Processed 40/244 batches | Batch Loss: 0.0042
Processed 60/244 batches | Batch Loss: 0.0025
Processed 80/244 batches | Batch Loss: 0.0046
Processed 100/244 batches | Batch Loss: 0.0022
Processed 120/244 batches | Batch Loss: 0.0028
Processed 140/244 batches | Batch Loss: 0.0018
Processed 160/244 batches | Batch Loss: 0.0018
Processed 180/244 batches | Batch Loss: 0.0036
Processed 200/244 batches | Batch Loss: 0.0075
Processed 220/244 batches | Batch Loss: 0.0075
Processed 240/244 batches | Batch Loss: 0.0579

Epoch 2 Training Loss: 0.0202


In [66]:
validation_loss, predictions, true_labels = evaluate_model(
    model,
    val_dataloader,
    loss_function
)
print(f"Epoch 2 Validation Loss: {validation_loss:.4f}")

Epoch 2 Validation Loss: 0.0390


In [51]:
accuracy = accuracy_score(true_labels, predictions)

precision = precision_score(true_labels, predictions, zero_division=0)

recall = recall_score(true_labels, predictions, zero_division=0)

f1 = f1_score(true_labels, predictions, zero_division=0)

In [52]:
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

Accuracy:  0.9916
Precision: 1.0000
Recall:    0.9375
F1-score:  0.9677


In [57]:
cm = confusion_matrix(true_labels, predictions)
print(cm)

[[724   0]
 [  7 105]]


In [58]:
training_loss = train_one_epoch(
    model,
    train_dataloader,
    optimizer,
    loss_function
)

print(f"\nEpoch 3 Training Loss: {training_loss:.4f}")

Processed 20/244 batches | Batch Loss: 0.0007
Processed 40/244 batches | Batch Loss: 0.0013
Processed 60/244 batches | Batch Loss: 0.0012
Processed 80/244 batches | Batch Loss: 0.0006
Processed 100/244 batches | Batch Loss: 0.0006
Processed 120/244 batches | Batch Loss: 0.0007
Processed 140/244 batches | Batch Loss: 0.0005
Processed 160/244 batches | Batch Loss: 0.0008
Processed 180/244 batches | Batch Loss: 0.0013
Processed 200/244 batches | Batch Loss: 0.1641
Processed 220/244 batches | Batch Loss: 0.0832
Processed 240/244 batches | Batch Loss: 0.0012

Epoch 3 Training Loss: 0.0074


In [65]:
validation_loss, predictions, true_labels = evaluate_model(
    model,
    val_dataloader,
    loss_function
)
print(f"Epoch 3 Validation Loss: {validation_loss:.4f}")

Epoch 3 Validation Loss: 0.0390


In [62]:
accuracy = accuracy_score(true_labels, predictions)

precision = precision_score(true_labels, predictions, zero_division=0)

recall = recall_score(true_labels, predictions, zero_division=0)

f1 = f1_score(true_labels, predictions, zero_division=0)

In [63]:
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

Accuracy:  0.9904
Precision: 0.9906
Recall:    0.9375
F1-score:  0.9633


In [64]:
cm = confusion_matrix(true_labels, predictions)
print(cm)

[[723   1]
 [  7 105]]


In [93]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [94]:
set_seed()

In [95]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [96]:
print(model.config.num_labels)

2


In [99]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

In [100]:
best_model_dir = Path("models/best_distilbert_cleaned")

In [101]:
best_f1 = -1.0

In [102]:
NUM_EPOCHS = 3
history = []

for epoch in range(1, NUM_EPOCHS + 1):

    print(f"\n{'=' * 50}")
    print(f"Epoch {epoch}/{NUM_EPOCHS}")
    print(f"{'=' * 50}")
    
    training_loss = train_one_epoch(
        model,
        train_dataloader,
        optimizer,
        loss_function
    )
    
    validation_loss, predictions, true_labels = evaluate_model(
        model,
        val_dataloader,
        loss_function
    )
    
    
    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions, zero_division=0)
    recall = recall_score(true_labels, predictions, zero_division=0)
    f1 = f1_score(true_labels, predictions, zero_division=0)

    cm = confusion_matrix(true_labels, predictions)
    
    epoch_results = {
        "epoch": epoch,
        "training_loss": training_loss,
        "validation_loss": validation_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm.tolist()
    }

    history.append(epoch_results)

    print(f"Training Loss:   {training_loss:.4f}")
    print(f"Validation Loss: {validation_loss:.4f}")
    print(f"Accuracy:        {accuracy:.4f}")
    print(f"Precision:       {precision:.4f}")
    print(f"Recall:          {recall:.4f}")
    print(f"F1 Score:        {f1:.4f}")

    print("\nConfusion Matrix:")
    print(cm)
    
    if f1 > best_f1:
        best_f1 = f1

        model.save_pretrained(str(best_model_dir))
        tokenizer.save_pretrained(str(best_model_dir))

        print(f"\nNew best model saved! F1: {best_f1:.4f}")


Epoch 1/3
Processed 20/227 batches | Batch Loss: 0.0762
Processed 40/227 batches | Batch Loss: 0.3498
Processed 60/227 batches | Batch Loss: 0.0808
Processed 80/227 batches | Batch Loss: 0.0227
Processed 100/227 batches | Batch Loss: 0.0060
Processed 120/227 batches | Batch Loss: 0.0149
Processed 140/227 batches | Batch Loss: 0.0098
Processed 160/227 batches | Batch Loss: 0.0122
Processed 180/227 batches | Batch Loss: 0.0087
Processed 200/227 batches | Batch Loss: 0.0078
Processed 220/227 batches | Batch Loss: 0.0583
Training Loss:   0.0879
Validation Loss: 0.0460
Accuracy:        0.9884
Precision:       0.9785
Recall:          0.9286
F1 Score:        0.9529

Confusion Matrix:
[[675   2]
 [  7  91]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


New best model saved! F1: 0.9529

Epoch 2/3
Processed 20/227 batches | Batch Loss: 0.2264
Processed 40/227 batches | Batch Loss: 0.0033
Processed 60/227 batches | Batch Loss: 0.0041
Processed 80/227 batches | Batch Loss: 0.0058
Processed 100/227 batches | Batch Loss: 0.0079
Processed 120/227 batches | Batch Loss: 0.0024
Processed 140/227 batches | Batch Loss: 0.0027
Processed 160/227 batches | Batch Loss: 0.0036
Processed 180/227 batches | Batch Loss: 0.0011
Processed 200/227 batches | Batch Loss: 0.0035
Processed 220/227 batches | Batch Loss: 0.0024
Training Loss:   0.0179
Validation Loss: 0.0567
Accuracy:        0.9819
Precision:       0.8962
Recall:          0.9694
F1 Score:        0.9314

Confusion Matrix:
[[666  11]
 [  3  95]]

Epoch 3/3
Processed 20/227 batches | Batch Loss: 0.0008
Processed 40/227 batches | Batch Loss: 0.0860
Processed 60/227 batches | Batch Loss: 0.0057
Processed 80/227 batches | Batch Loss: 0.0011
Processed 100/227 batches | Batch Loss: 0.0016
Processed 120/

In [103]:
for result in history:
    print(result)

{'epoch': 1, 'training_loss': 0.08788962394097946, 'validation_loss': 0.0460351400077343, 'accuracy': 0.9883870967741936, 'precision': 0.978494623655914, 'recall': 0.9285714285714286, 'f1': 0.9528795811518325, 'confusion_matrix': [[675, 2], [7, 91]]}
{'epoch': 2, 'training_loss': 0.017852563414279415, 'validation_loss': 0.056716273606063856, 'accuracy': 0.9819354838709677, 'precision': 0.8962264150943396, 'recall': 0.9693877551020408, 'f1': 0.9313725490196079, 'confusion_matrix': [[666, 11], [3, 95]]}
{'epoch': 3, 'training_loss': 0.016364846275309784, 'validation_loss': 0.05768994187215163, 'accuracy': 0.984516129032258, 'precision': 0.9215686274509803, 'recall': 0.9591836734693877, 'f1': 0.94, 'confusion_matrix': [[669, 8], [4, 94]]}


In [104]:
history_path = Path("models/training_history_cleaned.json")

with open(history_path, "w") as file:
    json.dump(history, file, indent=4)

In [105]:
print(history_path.exists())

True


In [106]:
for path in Path("models").rglob("*"):
    print(path)

models/best_distilbert_cleaned
models/training_history_cleaned.json
models/best_distilbert_cleaned/tokenizer_config.json
models/best_distilbert_cleaned/model.safetensors
models/best_distilbert_cleaned/tokenizer.json
models/best_distilbert_cleaned/config.json


In [107]:
best_model = AutoModelForSequenceClassification.from_pretrained(
    str(best_model_dir)
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

## Preparing model for testing

In [108]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_model.to(device)
best_model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [109]:
def evaluate_model(model, dataloader, loss_function, device):

    model.eval()

    total_loss = 0
    predictions = []
    true_labels = []

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = loss_function(outputs.logits, labels)

            total_loss += loss.item() * labels.size(0)

            batch_predictions = torch.argmax(outputs.logits, dim=1)

            predictions.extend(batch_predictions.cpu().tolist())
            true_labels.extend(labels.cpu().tolist())

    average_loss = total_loss / len(dataloader.dataset)

    return average_loss, predictions, true_labels

In [110]:
test_loss, test_predictions, test_true_labels = evaluate_model(
    best_model,
    test_dataloader,
    loss_function,
    device
)

In [111]:
test_accuracy = accuracy_score(test_true_labels, test_predictions)

test_precision = precision_score(
    test_true_labels, test_predictions, zero_division=0
)

test_recall = recall_score(
    test_true_labels, test_predictions, zero_division=0
)

test_f1 = f1_score(
    test_true_labels, test_predictions, zero_division=0
)

test_cm = confusion_matrix(test_true_labels, test_predictions)

In [121]:
print(f"Test Loss:      {test_loss:.4f}")
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print(f"Test F1 Score:  {test_f1:.4f}")

print("\nTest Confusion Matrix:")
print(test_cm)

print("\nClassification Report:")
print(
    classification_report(
        test_true_labels,
        test_predictions,
        target_names=["HAM", "SPAM"],
        zero_division=0
    )
)

Test Loss:      0.0297
Test Accuracy:  0.9936
Test Precision: 1.0000
Test Recall:    0.9490
Test F1 Score:  0.9738

Test Confusion Matrix:
[[678   0]
 [  5  93]]

Classification Report:
              precision    recall  f1-score   support

         HAM       0.99      1.00      1.00       678
        SPAM       1.00      0.95      0.97        98

    accuracy                           0.99       776
   macro avg       1.00      0.97      0.99       776
weighted avg       0.99      0.99      0.99       776



## Validation Error Analysis

### Objective
Examine the validation examples that DistilBERT misclassified to understand the model's weaknesses and identify possible areas for improvement.

### Why Validation Data?
We use the validation set for error analysis because it allows us to investigate model errors and guide future improvements without compromising the independence of our final test evaluation.

### Analysis Plan
1. Generate predictions on the validation dataset.
2. Compare predicted labels with actual labels.
3. Identify incorrectly classified SMS messages.
4. Examine the messages to understand possible causes of misclassification.
5. Document findings and potential improvements.

### Important Note
The test set remains untouched during model development. It will only be used for final evaluation after all model decisions have been completed.

**Label Mapping:**
- `0` = HAM
- `1` = SPAM


In [122]:
val_loss, val_predictions, val_true_labels = evaluate_model(
    best_model,
    val_dataloader,
    loss_function,
    device
)

In [118]:
error_analysis = pd.DataFrame({
    "message": X_val.tolist(),
    "actual": y_val.tolist(),
    "predicted": val_predictions
})

In [127]:
misclassified = error_analysis[
    error_analysis["actual"] != error_analysis["predicted"]
]

print(f"Misclassified messages: {len(misclassified)}")

misclassified["message_length"] = (
    misclassified["message"].str.len()
)

print(
    misclassified[
        ["message_length", "actual", "predicted", "message"]
    ].to_string(index=False)
)

Misclassified messages: 9
 message_length  actual  predicted                                                                                                                                                          message
             40       1          0                                                                                                                         RCT' THNQ Adrian for U text. Rgds Vatian
             72       1          0                                                                                         RECPT 1/3. You have ordered a Ringtone. Your order is being processed...
             64       0          1                                                                                                 Total video converter free download type this in google search:)
            157       1          0    Xmas & New Years Eve tickets are now on sale from the club, during the day from 10am till 8pm, and on Thurs, Fri & Sat night this week. They're selling 

In [124]:
print(misclassified.to_string(index=False))

                                                                                                                                                         message  actual  predicted
                                                                                                                        RCT' THNQ Adrian for U text. Rgds Vatian       1          0
                                                                                        RECPT 1/3. You have ordered a Ringtone. Your order is being processed...       1          0
                                                                                                Total video converter free download type this in google search:)       0          1
   Xmas & New Years Eve tickets are now on sale from the club, during the day from 10am till 8pm, and on Thurs, Fri & Sat night this week. They're selling fast!       1          0
As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertu

In [128]:
validation_analysis = error_analysis.copy()

validation_analysis["message_length"] = (
    validation_analysis["message"].str.len()
)

validation_analysis["result"] = (
    validation_analysis["actual"] ==
    validation_analysis["predicted"]
).map({
    True: "Correct",
    False: "Misclassified"
})

print(
    validation_analysis.groupby("result")["message_length"].agg(
        ["count", "mean", "median", "min", "max"]
    )
)

               count        mean  median  min  max
result                                            
Correct          766   80.182768    58.0    3  611
Misclassified      9  105.000000    84.0   40  160


In [129]:
misclassified_token_lengths = tokenizer(
    misclassified["message"].tolist(),
    truncation=False
)

token_lengths = [
    len(ids)
    for ids in misclassified_token_lengths["input_ids"]
]

misclassified["token_length"] = token_lengths

print(
    misclassified[
        ["message_length", "token_length", "message"]
    ].to_string(index=False)
)

 message_length  token_length                                                                                                                                                          message
             40            18                                                                                                                         RCT' THNQ Adrian for U text. Rgds Vatian
             72            23                                                                                         RECPT 1/3. You have ordered a Ringtone. Your order is being processed...
             64            15                                                                                                 Total video converter free download type this in google search:)
            157            46    Xmas & New Years Eve tickets are now on sale from the club, during the day from 10am till 8pm, and on Thurs, Fri & Sat night this week. They're selling fast!
            160            50 As per your req

In [130]:
misclassified_encodings = tokenizer(
    misclassified["message"].tolist(),
    truncation=True,
    max_length=128,
    padding=True,
    return_tensors="pt"
)

best_model.eval()

with torch.no_grad():
    outputs = best_model(
        input_ids=misclassified_encodings["input_ids"].to(device),
        attention_mask=misclassified_encodings["attention_mask"].to(device)
    )

probabilities = torch.softmax(outputs.logits, dim=1)

misclassified["ham_probability"] = (
    probabilities[:, 0].cpu().numpy()
)

misclassified["spam_probability"] = (
    probabilities[:, 1].cpu().numpy()
)

print(
    misclassified[
        ["actual", "predicted", "ham_probability", "spam_probability"]
    ].to_string(index=False)
)

 actual  predicted  ham_probability  spam_probability
      1          0         0.997959          0.002041
      1          0         0.993911          0.006089
      0          1         0.119573          0.880427
      1          0         0.767618          0.232382
      0          1         0.432132          0.567868
      1          0         0.944690          0.055310
      1          0         0.957401          0.042599
      1          0         0.948427          0.051573
      1          0         0.996705          0.003295


In [131]:
false_negatives = misclassified[
    (misclassified["actual"] == 1) &
    (misclassified["predicted"] == 0)
]

print(
    false_negatives[
        ["message", "ham_probability", "spam_probability"]
    ].to_string(index=False)
)

                                                                                                                                                      message  ham_probability  spam_probability
                                                                                                                     RCT' THNQ Adrian for U text. Rgds Vatian         0.997959          0.002041
                                                                                     RECPT 1/3. You have ordered a Ringtone. Your order is being processed...         0.993911          0.006089
Xmas & New Years Eve tickets are now on sale from the club, during the day from 10am till 8pm, and on Thurs, Fri & Sat night this week. They're selling fast!         0.767618          0.232382
                                                                         Would you like to see my XXX pics they are so hot they were nearly banned in the uk!         0.944690          0.055310
                                   

In [132]:
false_positives = misclassified[
    (misclassified["actual"] == 0) &
    (misclassified["predicted"] == 1)
]

print(
    false_positives[
        ["message", "ham_probability", "spam_probability"]
    ].to_string(index=False)
)

                                                                                                                                                         message  ham_probability  spam_probability
                                                                                                Total video converter free download type this in google search:)         0.119573          0.880427
As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune         0.432132          0.567868


In [134]:
test_analysis = pd.DataFrame({
    "message": X_test.tolist(),
    "actual": test_true_labels,
    "predicted": test_predictions
})

test_misclassified = test_analysis[
    test_analysis["actual"] != test_analysis["predicted"]
].copy()

print(f"Test errors: {len(test_misclassified)}")

print(test_misclassified.to_string(index=False))

Test errors: 5
                                                                                                                                      message  actual  predicted
                                                                                          Money i have won wining number 946 wot do i do next       1          0
Burger King - Wanna play footy at a top stadium? Get 2 Burger King before 1st Sept and go Large or Super with Coca-Cola and walk out a winner       1          0
     New Tones This week include: 1)McFly-All Ab.., 2) Sara Jorge-Shock.. 3) Will Smith-Switch.. To order follow instructions on next message       1          0
                                   Oh my god! I've found your number again! I'm so glad, text me back xafter this msgs cst std ntwk chg £1.50       1          0
       Hello. We need some posh birds and chaps to user trial prods for champneys. Can i put you down? I need your address and dob asap. Ta r       1          0


In [135]:
test_error_encodings = tokenizer(
    test_misclassified["message"].tolist(),
    truncation=True,
    max_length=128,
    padding=True,
    return_tensors="pt"
)

best_model.eval()

with torch.no_grad():
    outputs = best_model(
        input_ids=test_error_encodings["input_ids"].to(device),
        attention_mask=test_error_encodings["attention_mask"].to(device)
    )

probabilities = torch.softmax(outputs.logits, dim=1)

test_misclassified["ham_probability"] = (
    probabilities[:, 0].cpu().numpy()
)

test_misclassified["spam_probability"] = (
    probabilities[:, 1].cpu().numpy()
)

print(
    test_misclassified[
        ["actual", "predicted", "ham_probability", "spam_probability"]
    ].to_string(index=False)
)

 actual  predicted  ham_probability  spam_probability
      1          0         0.997386          0.002614
      1          0         0.738068          0.261932
      1          0         0.888848          0.111152
      1          0         0.501089          0.498911
      1          0         0.997526          0.002474


In [136]:
#calculate probability for all val set 

best_model.eval()

all_val_probabilities = []
all_val_labels = []

with torch.no_grad():
    for batch in val_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        outputs = best_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(outputs.logits, dim=1)

        all_val_probabilities.extend(
            probabilities[:, 1].cpu().numpy()
        )

        all_val_labels.extend(labels.numpy())

In [137]:
print("Validation probabilities:", len(all_val_probabilities))
print("Validation labels:", len(all_val_labels))

print("\nFirst 5 spam probabilities:")
print(all_val_probabilities[:5])

print("\nFirst 5 actual labels:")
print(all_val_labels[:5])

Validation probabilities: 775
Validation labels: 775

First 5 spam probabilities:
[np.float32(0.0022263308), np.float32(0.97941804), np.float32(0.0020409042), np.float32(0.0021689248), np.float32(0.0019816416)]

First 5 actual labels:
[np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0)]


## Validation Error Analysis — Findings

The DistilBERT model misclassified 6 out of 836 validation examples.

### False Positives (4)
Four legitimate HAM messages were incorrectly classified as SPAM. These messages included commercial notifications, payment references, website mentions, and unfamiliar conversational contexts.

### False Negatives (2)
Two SPAM messages were incorrectly classified as HAM. One used abbreviated SMS language, while the other used conversational and flattering language.

### Key Observations
1. Commercial vocabulary does not necessarily indicate spam.
2. Legitimate messages can contain payment references and website mentions.
3. Spam messages do not always contain obvious promotional or financial keywords.
4. Context and language patterns remain important challenges for text classification.

### Conclusion
The model achieved strong validation performance, but these errors demonstrate that high classification accuracy does not imply perfect classification. Further improvements should be evaluated using validation data without compromising the independent test set.